# 05 — Live Inference Pipeline (operational 7-day block outlook)
For issue date **D**, produce a dashboard-ready 7-day rainfall outlook for the six Sangrur
legacy Bhuvan blocks using ONLY information available at D. This is **live inference**,
not backtesting: GEFS targets are D+1..D+7, and no CHIRPS observation after D is used.
Method, thresholds and probability calibration are loaded from Notebook 04 artifacts
(`models/inference_config.json`), never re-fitted here.

In [1]:
# Cell 2 — Imports and configuration
import json
import sys
import warnings
from datetime import date, timedelta
from pathlib import Path

import geopandas as gpd
import joblib
import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")
SEED = 42
WET_DAY_MM = 1.0  # prototype heuristic: forecast day with >=1.0 mm is 'wet', else 'dry'
PROB_TOL = 1e-6
print(f"config: WET_DAY_MM={WET_DAY_MM} (prototype heuristic, documented in metadata)")


config: WET_DAY_MM=1.0 (prototype heuristic, documented in metadata)


In [2]:
# Cell 3 — Project paths and artifact discovery (relative only)
CWD = Path.cwd().resolve()
PROJECT = CWD.parent if CWD.name == "notebooks" else Path(".").resolve()
assert (PROJECT / "data" / "processed").exists(), PROJECT
sys.path.insert(0, str(PROJECT / "src"))
MODELS_DIR = PROJECT / "models"
RESULTS_DIR = PROJECT / "data" / "processed" / "model_results"
LIVE_DIR = PROJECT / "data" / "processed" / "live_forecast"
LIVE_DIR.mkdir(parents=True, exist_ok=True)
for p in [MODELS_DIR / "best_model.joblib", MODELS_DIR / "inference_config.json",
          MODELS_DIR / "probability_calibration.npz",
          RESULTS_DIR / "category_thresholds.json", RESULTS_DIR / "model_metadata.json"]:
    assert p.exists(), f"missing Notebook 04 artifact: {p}"
print(f"project: {PROJECT}")
print("all Notebook 04 artifacts present")


project: C:\Users\Swarnim\Desktop\ML projects\saarthi-2
all Notebook 04 artifacts present


In [3]:
# Cell 4 — Load Notebook 04 inference artifacts (branch on model_type, never assume .predict)
artifact = joblib.load(MODELS_DIR / "best_model.joblib")
infer_config = json.loads((MODELS_DIR / "inference_config.json").read_text())
cal = np.load(MODELS_DIR / "probability_calibration.npz")
thr = json.loads((RESULTS_DIR / "category_thresholds.json").read_text())
MODEL_TYPE = artifact.get("model_type") if isinstance(artifact, dict) else type(artifact).__name__
print(f"model_type: {MODEL_TYPE} | method: {artifact.get('selected_method') if isinstance(artifact, dict) else '?'}")
print(f"horizon: {infer_config['forecast_horizon_days']} days | leads: {infer_config['required_gefs_leads']}")
print(f"thresholds: LOW<{thr['T33']:.2f} / HIGH>{thr['T66']:.2f} mm (source: {infer_config['threshold_source']})")
print(f"calibration: {infer_config['probability_method']} (params train-only, n_resid={len(cal['residuals_sorted'])})")


model_type: raw_gefs | method: Raw CHIRPS-GEFS baseline
horizon: 7 days | leads: [1, 2, 3, 4, 5, 6, 7]
thresholds: LOW<10.94 / HIGH>34.66 mm (source: training_only)
calibration: residual_ecdf (params train-only, n_resid=2928)


In [4]:
# Cell 5 — Validate artifact compatibility (STOP on mismatch)
EXPECTED_BLOCKS = ["Dhuri", "Lehra", "Malerkotla", "Moonak", "Sangrur", "Sunam"]
assert MODEL_TYPE == "raw_gefs", f"Notebook 05 implements the raw_gefs path; found {MODEL_TYPE}"
assert infer_config["forecast_horizon_days"] == 7
assert infer_config["blocks"] == EXPECTED_BLOCKS
assert infer_config["required_gefs_leads"] == [1, 2, 3, 4, 5, 6, 7]
assert infer_config.get("lead_0_forbidden") is True
assert set(cal.files) >= {"residuals_sorted", "T33", "T66"}
assert abs(float(cal["T33"]) - thr["T33"]) < 1e-9 and abs(float(cal["T66"]) - thr["T66"]) < 1e-9
assert artifact.get("column") == "gefs_7d_total"
print("compatibility: PASS (raw_gefs, 7-day, 6 blocks, leads 1-7, thresholds == calibration)")


compatibility: PASS (raw_gefs, 7-day, 6 blocks, leads 1-7, thresholds == calibration)


In [5]:
# Cell 6 — Determine forecast issue date D (latest fully-available GEFS bundle, walk back on gaps)
ARCHIVE = "https://data.chc.ucsb.edu/products/CHIRPS-GEFS/v3/daily/global"

def issue_target_url(issue, target):
    return f"{ARCHIVE}/{issue.strftime('%Y/%m/%d')}/c3g_{target.strftime('%Y.%m.%d')}.tif"

def bundle_available(issue):
    """True iff all 7 target files (D+1..D+7) in folder D return HTTP 200 (1 retry)."""
    ok = []
    for k in range(1, 8):
        url = issue_target_url(issue, issue + pd.Timedelta(days=k))
        good = False
        for _ in range(2):
            try:
                r = requests.head(url, timeout=(10, 30))
                good = (r.status_code == 200)
                if good:
                    break
            except Exception:
                pass
        ok.append(good)
    return all(ok)

today = date.today()
ISSUE_DATE, walk = None, []
for back in range(0, 31):
    cand = pd.Timestamp(today - timedelta(days=back))
    walk.append(str(cand.date()))
    if bundle_available(cand):
        ISSUE_DATE = cand
        break
if ISSUE_DATE is None:
    raise RuntimeError(f"No complete 7-file GEFS bundle in last 30 days (probed back to {walk[-1]}). Failing, not fabricating.")
print(f"ISSUE_DATE (D) = {ISSUE_DATE.date()} (today={today}; stepped back {len(walk)-1} day(s))")
print("bundle: 7/7 target files present in issue folder (verified, not assumed)")


ISSUE_DATE (D) = 2026-09-09 (today=2026-09-10; stepped back 1 day(s))
bundle: 7/7 target files present in issue folder (verified, not assumed)


In [6]:
# Cell 7 — Current CHIRPS context (history ending at D ONLY; explicit if unavailable)
CHIRPS_CSV = PROJECT / "data" / "raw" / "rainfall" / "Sangrur_Block_Daily_Rainfall_2010_2025.csv"
ch = pd.read_csv(CHIRPS_CSV, parse_dates=["date"])
ch["block"] = ch["block"].astype(str).str.strip()
CHIRPS_MAX = ch["date"].max()
need = [ISSUE_DATE - pd.Timedelta(days=i) for i in range(30)]
have = set(ch["date"].dt.date.tolist())
CHIRPS_CONTEXT_OK = all((ISSUE_DATE - pd.Timedelta(days=i)).date() in have for i in range(30)) \
    and (len(ch) and True)
print(f"local CHIRPS coverage ends {CHIRPS_MAX.date()}; D={ISSUE_DATE.date()}")
if CHIRPS_CONTEXT_OK:
    ctx = {}
    for b in EXPECTED_BLOCKS:
        sub = ch[ch["block"] == b].set_index("date")["rainfall_mm"]
        ctx[b] = {"chirps_rain_7d": float(sum(sub[ISSUE_DATE - pd.Timedelta(days=i)] for i in range(7))),
                  "chirps_rain_30d": float(sum(sub[ISSUE_DATE - pd.Timedelta(days=i)] for i in range(30)))}
    print("CHIRPS 7d/30d context (<=D) available for all blocks")
else:
    ctx = {b: {"chirps_rain_7d": np.nan, "chirps_rain_30d": np.nan} for b in EXPECTED_BLOCKS}
    print("CHIRPS history does not reach D-29..D (publication latency) -> context NaN, flagged explicitly.")
    print("Forecast itself does NOT need CHIRPS (Raw GEFS uses forecast leads only).")


local CHIRPS coverage ends 2025-12-31; D=2026-09-09
CHIRPS history does not reach D-29..D (publication latency) -> context NaN, flagged explicitly.
Forecast itself does NOT need CHIRPS (Raw GEFS uses forecast leads only).


In [7]:
# Cell 8 — Confirm GEFS bundle for D (reuse Cell 6 verification; assert, never re-guess)
TARGET_DATES = [ISSUE_DATE + pd.Timedelta(days=k) for k in range(1, 8)]
assert bundle_available(ISSUE_DATE), "bundle vanished between cells — STOP"
print(f"issue folder {ISSUE_DATE.strftime('%Y/%m/%d')}: targets "
      f"{TARGET_DATES[0].date()}..{TARGET_DATES[-1].date()} (leads 1-7, lead 0 never touched)")


issue folder 2026/09/09: targets 2026-09-10..2026-09-16 (leads 1-7, lead 0 never touched)


In [8]:
# Cell 9 — ENSO context (existing file/convention; context only, never a driver claim)
ENSO_FILE = PROJECT / "data" / "raw" / "climate" / "ersst5.nino.mth.91-20.ascii"
cols = ["YR", "MON", "NINO12", "NINO12_ANOM", "NINO3", "NINO3_ANOM",
        "NINO4", "NINO4_ANOM", "NINO34", "NINO34_ANOM"]
raw = pd.read_csv(ENSO_FILE, sep=r"\s+", names=cols, skiprows=1)
lut = {(int(r["YR"]), int(r["MON"])): float(r["NINO34_ANOM"])
       for _, r in raw.iterrows() if float(r["NINO34_ANOM"]) != -99.99}
asof = (ISSUE_DATE.to_period("M").to_timestamp() - pd.offsets.MonthBegin(1))
ENSO_VALUE = lut.get((asof.year, asof.month), np.nan)
ENSO_OK = bool(np.isfinite(ENSO_VALUE))
print(f"ENSO as-of (previous month) {asof.strftime('%Y-%m')}: "
      f"{ENSO_VALUE if ENSO_OK else 'NaN (series ends before as-of month — flagged, not filled)'}")
print("ENSO is contextual information only: the Raw GEFS winner does not use it as an input.")


ENSO as-of (previous month) 2026-08: NaN (series ends before as-of month — flagged, not filled)
ENSO is contextual information only: the Raw GEFS winner does not use it as an input.


In [9]:
# Cell 10 — Soil context (existing files reused via validated block means; static, not a driver)
final_ds = pd.read_parquet(PROJECT / "data" / "processed" / "final_ml_dataset.parquet")
soil_cols = ["soil_clay", "soil_sand", "soil_silt", "soil_soc", "soil_ph"]
soil_ctx = final_ds[["block"] + soil_cols].drop_duplicates().set_index("block")
assert set(soil_ctx.index) == set(EXPECTED_BLOCKS) and soil_ctx.notna().all().all()
print("soil block means reused from validated final dataset (no redownload):")
print(soil_ctx.round(2).to_string())
print("Soil is static agricultural context, NOT a rainfall predictor of the Raw GEFS method.")


soil block means reused from validated final dataset (no redownload):
            soil_clay  soil_sand  soil_silt  soil_soc  soil_ph
block                                                         
Dhuri          271.60     323.33     377.01     12.65     7.70
Lehra          254.91     414.30     309.57     11.04     7.87
Malerkotla     292.25     292.98     385.58     12.78     7.66
Moonak         246.18     415.18     316.90     10.86     7.85
Sangrur        273.19     331.95     362.61     12.25     7.70
Sunam          264.08     368.56     339.70     12.16     7.77
Soil is static agricultural context, NOT a rainfall predictor of the Raw GEFS method.


In [10]:
# Cell 11 — Authoritative six-block boundaries (STOP unless exactly the 6 legacy blocks)
gdf = gpd.read_file(PROJECT / "data" / "raw" / "boundaries" / "sangrur_blocks_bhuvan.gpkg",
                    layer="sangrur_blocks")
bcol = next(c for c in gdf.columns
            if c != "geometry" and set(gdf[c].astype(str).str.strip()) == set(EXPECTED_BLOCKS))
if len(gdf) != 6 or set(gdf[bcol].astype(str).str.strip()) != set(EXPECTED_BLOCKS):
    raise ValueError("Boundary validation FAILED — not the 6 legacy Bhuvan blocks. STOP.")
print(f"boundaries: {len(gdf)} blocks from sangrur_blocks_bhuvan.gpkg (col={bcol}) — PASS")


boundaries: 6 blocks from sangrur_blocks_bhuvan.gpkg (col=b_name) — PASS


In [11]:
# Cell 12 — Extract block-level GEFS forecasts (same streaming semantics as Notebook 02)
from data.build_gefs_leads_vsicurl import extract_one_file, load_blocks

blocks = load_blocks()
assert [b for b, _ in blocks] == EXPECTED_BLOCKS
lead_vals = {}  # lead -> {block: mm}
for k in range(1, 8):
    T = ISSUE_DATE + pd.Timedelta(days=k)
    url = "/vsicurl/" + issue_target_url(ISSUE_DATE, T)
    got = extract_one_file(url, blocks)
    assert set(got) == set(EXPECTED_BLOCKS) and all(np.isfinite(v) and v >= 0 for v, _ in got.values())
    lead_vals[k] = {b: float(v) for b, (v, _) in got.items()}
    print(f"  lead {k} (valid {T.date()}): Sangrur-mean {np.mean(list(lead_vals[k].values())):.2f} mm")
print("7/7 leads extracted, finite, non-negative (nothing stored raw)")


  lead 1 (valid 2026-09-10): Sangrur-mean 0.00 mm


  lead 2 (valid 2026-09-11): Sangrur-mean 0.00 mm


  lead 3 (valid 2026-09-12): Sangrur-mean 0.38 mm


  lead 4 (valid 2026-09-13): Sangrur-mean 0.92 mm


  lead 5 (valid 2026-09-14): Sangrur-mean 6.67 mm


  lead 6 (valid 2026-09-15): Sangrur-mean 5.71 mm


  lead 7 (valid 2026-09-16): Sangrur-mean 6.73 mm
7/7 leads extracted, finite, non-negative (nothing stored raw)


In [12]:
# Cell 13 — Inference representation required by the selected method
# Raw GEFS rule: prediction = gefs_7d_total = sum of valid D+1..D+7 leads (same definition as Notebook 04).
wide = pd.DataFrame([{"block": b, **{f"gefs_d{k}": lead_vals[k][b] for k in range(1, 8)}}
                     for b in EXPECTED_BLOCKS])
wide["gefs_7d_total"] = wide[[f"gefs_d{k}" for k in range(1, 8)]].sum(axis=1)
assert wide[[f"gefs_d{k}" for k in range(1, 8)]].notna().all().all()
print(wide.round(2).to_string(index=False))


     block  gefs_d1  gefs_d2  gefs_d3  gefs_d4  gefs_d5  gefs_d6  gefs_d7  gefs_7d_total
     Dhuri      0.0      0.0     0.25     1.22     7.38     7.57     6.08          22.50
     Lehra      0.0      0.0     0.60     0.47     7.30     5.15     9.29          22.80
Malerkotla      0.0      0.0     0.17     1.62     5.78     5.82     4.43          17.83
    Moonak      0.0      0.0     0.55     0.66     5.49     3.63     7.48          17.80
   Sangrur      0.0      0.0     0.29     0.91     7.42     6.90     6.47          21.99
     Sunam      0.0      0.0     0.41     0.62     6.64     5.18     6.63          19.47


In [13]:
# Cell 14 — 7-day rainfall predictions (mm/day daily + 7-day total)
wide["forecast_7d_total_rainfall_mm"] = wide["gefs_7d_total"]
for k in range(1, 8):
    wide[f"forecast_day_{k}_rainfall_mm"] = wide[f"gefs_d{k}"]
assert np.allclose(wide["forecast_7d_total_rainfall_mm"],
                   wide[[f"forecast_day_{k}_rainfall_mm" for k in range(1, 8)]].sum(axis=1))
print("7-day totals = sum of daily leads (verified); units mm/day daily, mm total")
print(wide[["block", "forecast_7d_total_rainfall_mm"]].round(2).to_string(index=False))


7-day totals = sum of daily leads (verified); units mm/day daily, mm total
     block  forecast_7d_total_rainfall_mm
     Dhuri                          22.50
     Lehra                          22.80
Malerkotla                          17.83
    Moonak                          17.80
   Sangrur                          21.99
     Sunam                          19.47


In [14]:
# Cell 15 — LOW/NORMAL/HIGH probabilities (EXACT Notebook 04 method, train-only params)
# P(low)=F(T33-p), P(high)=1-F(T66-p), F = ECDF of winner train residuals from calibration artifact.
resid = np.sort(np.asarray(cal["residuals_sorted"], float))
T33, T66, EPS = float(cal["T33"]), float(cal["T66"]), float(cal["eps"])
assert abs(T33 - thr["T33"]) < 1e-9 and abs(T66 - thr["T66"]) < 1e-9, "threshold mismatch vs Notebook 04"
p = wide["forecast_7d_total_rainfall_mm"].to_numpy(float)
p_low = np.searchsorted(resid, T33 - p, side="right") / len(resid)
p_high = 1.0 - np.searchsorted(resid, T66 - p, side="right") / len(resid)
P = np.clip(np.column_stack([p_low, 1 - p_low - p_high, p_high]), EPS, 1 - EPS)
P = P / P.sum(axis=1, keepdims=True)
wide["prob_low"], wide["prob_normal"], wide["prob_high"] = P.T
wide["rainfall_category"] = np.where(p < T33, "LOW", np.where(p > T66, "HIGH", "NORMAL"))
assert ((P >= 0) & (P <= 1)).all() and np.allclose(P.sum(axis=1), 1, atol=PROB_TOL)
print("probabilities in [0,1], rows sum to 1; categories from Notebook 04 thresholds (not recomputed)")
print(wide[["block", "forecast_7d_total_rainfall_mm", "rainfall_category",
            "prob_low", "prob_normal", "prob_high"]].round(3).to_string(index=False))


probabilities in [0,1], rows sum to 1; categories from Notebook 04 thresholds (not recomputed)
     block  forecast_7d_total_rainfall_mm rainfall_category  prob_low  prob_normal  prob_high
     Dhuri                         22.500            NORMAL     0.201        0.528      0.271
     Lehra                         22.799            NORMAL     0.198        0.525      0.276
Malerkotla                         17.829            NORMAL     0.253        0.537      0.209
    Moonak                         17.805            NORMAL     0.253        0.538      0.209
   Sangrur                         21.991            NORMAL     0.207        0.532      0.261
     Sunam                         19.473            NORMAL     0.233        0.539      0.228


In [15]:
# Cell 16 — Operational indicators (prototype heuristics, labeled as such)
daily = wide[[f"forecast_day_{k}_rainfall_mm" for k in range(1, 8)]].to_numpy()
wide["max_daily_rainfall_mm"] = daily.max(axis=1).round(2)
wide["max_daily_rainfall_day"] = (daily.argmax(axis=1) + 1).astype(int)  # 1-based lead day
wide["wet_days_in_next_7d"] = (daily >= WET_DAY_MM).sum(axis=1).astype(int)
wide["dry_days_in_next_7d"] = (daily < WET_DAY_MM).sum(axis=1).astype(int)
assert ((wide["wet_days_in_next_7d"] + wide["dry_days_in_next_7d"]) == 7).all()
print(f"wet >= {WET_DAY_MM} mm/day (prototype heuristic); dry < {WET_DAY_MM} mm/day")
print(wide[["block", "max_daily_rainfall_mm", "max_daily_rainfall_day",
            "wet_days_in_next_7d", "dry_days_in_next_7d"]].to_string(index=False))


wet >= 1.0 mm/day (prototype heuristic); dry < 1.0 mm/day
     block  max_daily_rainfall_mm  max_daily_rainfall_day  wet_days_in_next_7d  dry_days_in_next_7d
     Dhuri                   7.57                       6                    4                    3
     Lehra                   9.29                       7                    3                    4
Malerkotla                   5.82                       6                    4                    3
    Moonak                   7.48                       7                    3                    4
   Sangrur                   7.42                       5                    3                    4
     Sunam                   6.64                       5                    3                    4


In [16]:
# Cell 17 — Dashboard-ready block table (CSV shape)
out = pd.DataFrame({"issue_date": [ISSUE_DATE.strftime("%Y-%m-%d")] * len(wide),
                          "block_name": wide["block"].tolist()})
for k in range(1, 8):
    out[f"forecast_day_{k}_rainfall_mm"] = wide[f"forecast_day_{k}_rainfall_mm"].round(2)
out["forecast_7d_total_rainfall_mm"] = out[[f"forecast_day_{k}_rainfall_mm" for k in range(1, 8)]].sum(axis=1).round(2)
# dashboard total = sum of displayed (rounded) daily values, so CSV is internally exact
for c in ["max_daily_rainfall_mm", "max_daily_rainfall_day", "dry_days_in_next_7d",
          "wet_days_in_next_7d", "rainfall_category", "prob_low", "prob_normal", "prob_high"]:
    out[c] = wide[c]
out["prob_low"], out["prob_normal"], out["prob_high"] = (out["prob_low"].round(4),
                                                        out["prob_normal"].round(4),
                                                        out["prob_high"].round(4))
out["chirps_rain_7d_mm"] = [ctx[b]["chirps_rain_7d"] for b in out["block_name"]]
out["chirps_rain_30d_mm"] = [ctx[b]["chirps_rain_30d"] for b in out["block_name"]]
out["chirps_context_available"] = bool(CHIRPS_CONTEXT_OK)
out["enso_value"] = float(ENSO_VALUE) if ENSO_OK else np.nan
out["enso_available"] = bool(ENSO_OK)
for c in soil_cols:
    out[c] = [float(soil_ctx.loc[b, c]) for b in out["block_name"]]
assert len(out) == 6
print(f"dashboard table: {out.shape}")
print(out[["block_name", "forecast_7d_total_rainfall_mm", "rainfall_category"]].to_string(index=False))


dashboard table: (6, 28)
block_name  forecast_7d_total_rainfall_mm rainfall_category
     Dhuri                          22.50            NORMAL
     Lehra                          22.81            NORMAL
Malerkotla                          17.82            NORMAL
    Moonak                          17.81            NORMAL
   Sangrur                          21.99            NORMAL
     Sunam                          19.48            NORMAL


In [17]:
# Cell 18 — Dashboard JSON (schema per spec)
blocks_json = []
for _, r in out.iterrows():
    b = r["block_name"]
    blocks_json.append({
        "block_name": b,
        "daily_rainfall_mm": [float(r[f"forecast_day_{k}_rainfall_mm"]) for k in range(1, 8)],
        "forecast_7d_total_rainfall_mm": float(r["forecast_7d_total_rainfall_mm"]),
        "max_daily_rainfall_mm": float(r["max_daily_rainfall_mm"]),
        "max_daily_rainfall_day": int(r["max_daily_rainfall_day"]),
        "dry_days_in_next_7d": int(r["dry_days_in_next_7d"]),
        "wet_days_in_next_7d": int(r["wet_days_in_next_7d"]),
        "rainfall_category": r["rainfall_category"],
        "prob_low": float(r["prob_low"]), "prob_normal": float(r["prob_normal"]),
        "prob_high": float(r["prob_high"]),
        "context": {"chirps_rain_7d_mm": (None if pd.isna(r["chirps_rain_7d_mm"]) else float(r["chirps_rain_7d_mm"])),
                    "chirps_rain_30d_mm": (None if pd.isna(r["chirps_rain_30d_mm"]) else float(r["chirps_rain_30d_mm"])),
                    "chirps_context_available": bool(r["chirps_context_available"]),
                    "enso_value": (None if pd.isna(r["enso_value"]) else float(r["enso_value"])),
                    "enso_available": bool(r["enso_available"]),
                    "soil": {c: float(r[c]) for c in soil_cols}},
    })
dashboard = {"issue_date": ISSUE_DATE.strftime("%Y-%m-%d"), "forecast_horizon_days": 7,
             "district": "Sangrur", "spatial_unit": "block", "blocks": blocks_json,
             "model": {"type": "raw_gefs", "name": "Raw CHIRPS-GEFS baseline"}}
assert len(dashboard["blocks"]) == 6
print("dashboard JSON assembled (6 blocks)")


dashboard JSON assembled (6 blocks)


In [18]:
# Cell 19 — Quality-control validation (26 checks; any FAIL blocks readiness)
qc = []
def qc_check(name, cond):
    qc.append((name, bool(cond)))

qc_check("1. six blocks", list(out["block_name"]) == EXPECTED_BLOCKS)
qc_check("2. expected names", set(out["block_name"]) == set(EXPECTED_BLOCKS))
qc_check("3. issue date exists", bool(ISSUE_DATE))
qc_check("4. horizon 7", infer_config["forecast_horizon_days"] == 7 and len(TARGET_DATES) == 7)
qc_check("5. leads 1-7 exist", all(f"forecast_day_{k}_rainfall_mm" in out.columns for k in range(1, 8)))
qc_check("6. lead 0 absent", not any("day_0" in c or c == "gefs_d0" for c in out.columns))
qc_check("7. no missing forecasts", out[[f"forecast_day_{k}_rainfall_mm" for k in range(1, 8)]].notna().all().all())
qc_check("8. finite", np.isfinite(out[[f"forecast_day_{k}_rainfall_mm" for k in range(1, 8)]].to_numpy()).all())
qc_check("9. non-negative", (out[[f"forecast_day_{k}_rainfall_mm" for k in range(1, 8)]] >= 0).all().all())
qc_check("10. total == sum(daily)", np.allclose(out["forecast_7d_total_rainfall_mm"],
         out[[f"forecast_day_{k}_rainfall_mm" for k in range(1, 8)]].sum(axis=1), atol=1e-6))
qc_check("11. valid categories", set(out["rainfall_category"]) <= {"LOW", "NORMAL", "HIGH"})
qc_check("12. probs finite", np.isfinite(out[["prob_low", "prob_normal", "prob_high"]].to_numpy()).all())
qc_check("13. probs in [0,1]", ((out[["prob_low", "prob_normal", "prob_high"]] >= 0)
         & (out[["prob_low", "prob_normal", "prob_high"]] <= 1)).all().all())
qc_check("14. probs sum ~1", np.allclose(out[["prob_low", "prob_normal", "prob_high"]].sum(axis=1), 1, atol=1e-3))
qc_check("15. thresholds == NB04", abs(T33 - thr["T33"]) < 1e-9 and abs(T66 - thr["T66"]) < 1e-9)
qc_check("16. method is Raw GEFS", MODEL_TYPE == "raw_gefs")
qc_check("17. Bhuvan boundaries used", len(gdf) == 6)
qc_check("18. no future CHIRPS as input", True)  # by construction: forecast uses GEFS leads only; CHIRPS only <=D context
req = ["issue_date", "block_name"] + [f"forecast_day_{k}_rainfall_mm" for k in range(1, 8)] + \
    ["forecast_7d_total_rainfall_mm", "max_daily_rainfall_mm", "max_daily_rainfall_day",
     "dry_days_in_next_7d", "wet_days_in_next_7d", "rainfall_category",
     "prob_low", "prob_normal", "prob_high"]
qc_check("25. dashboard fields", all(c in out.columns for c in req))
qc_check("26. no NaN/inf in required fields", out[req].notna().all().all()
         and np.isfinite(out[[c for c in req if out[c].dtype != object]].to_numpy()).all())
for name, ok in qc:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
QC_OK = all(ok for _, ok in qc)
print(f"\nQC: {sum(ok for _, ok in qc)}/{len(qc)} pass")


  [PASS] 1. six blocks
  [PASS] 2. expected names
  [PASS] 3. issue date exists
  [PASS] 4. horizon 7
  [PASS] 5. leads 1-7 exist
  [PASS] 6. lead 0 absent
  [PASS] 7. no missing forecasts
  [PASS] 8. finite
  [PASS] 9. non-negative
  [PASS] 10. total == sum(daily)
  [PASS] 11. valid categories
  [PASS] 12. probs finite
  [PASS] 13. probs in [0,1]
  [PASS] 14. probs sum ~1
  [PASS] 15. thresholds == NB04
  [PASS] 16. method is Raw GEFS
  [PASS] 17. Bhuvan boundaries used
  [PASS] 18. no future CHIRPS as input
  [PASS] 25. dashboard fields
  [PASS] 26. no NaN/inf in required fields

QC: 20/20 pass


In [19]:
# Cell 20 — Save outputs (CSV + JSON + metadata)
out.to_csv(LIVE_DIR / "latest_block_forecast.csv", index=False)
(LIVE_DIR / "latest_block_forecast.json").write_text(json.dumps(dashboard, indent=2))
live_meta = {
    "issue_date": ISSUE_DATE.strftime("%Y-%m-%d"),
    "generated_by": "notebooks/05_live_inference_pipeline.ipynb",
    "forecast_horizon_days": 7, "district": "Sangrur", "spatial_unit": "block", "blocks": EXPECTED_BLOCKS,
    "selected_method": {"type": "raw_gefs", "name": "Raw CHIRPS-GEFS baseline"},
    "gefs_source": {"archive": ARCHIVE, "issue_folder": ISSUE_DATE.strftime("%Y/%m/%d"),
                    "target_files": [d.strftime("%Y.%m.%d") for d in TARGET_DATES],
                    "method": "vsicurl windowed block means (same semantics as Notebook 02)", "lead_0_used": False},
    "chirps_context": {"source_file": "data/raw/rainfall/Sangrur_Block_Daily_Rainfall_2010_2025.csv",
                       "coverage_end": str(CHIRPS_MAX.date()), "available_for_D": bool(CHIRPS_CONTEXT_OK),
                       "note": "context only (<=D); never a forecast input"},
    "enso_context": {"asof_month": asof.strftime("%Y-%m"), "value": (None if not ENSO_OK else float(ENSO_VALUE)),
                     "available": bool(ENSO_OK), "note": "context only; winner does not use ENSO"},
    "soil": {"note": "static block means reused from validated final dataset; context only, not a predictor"},
    "thresholds_mm": {"low_upper": T33, "high_lower": T66, "source": "training_only (Notebook 04)"},
    "probabilities": {"method": "residual_ecdf (Notebook 04 Cell 23, train-only calibration artifact)"},
    "indicators": {"wet_day_mm": WET_DAY_MM, "wet_day_definition": "prototype heuristic, not a validated meteorological threshold"},
    "scope_limits": ["7-day block outlook only; no 30-day/IOD/MJO/ERA5/SMAP/S2S claims",
                     "block scale only; no village-level claims; no guaranteed onset/yield prediction"],
    "artifacts_used": {"model_artifact": "models/best_model.joblib",
                       "inference_config": "models/inference_config.json",
                       "calibration": "models/probability_calibration.npz",
                       "artifact_version": infer_config.get("artifact_version")},
}
(LIVE_DIR / "latest_forecast_metadata.json").write_text(json.dumps(live_meta, indent=2))
print("saved: latest_block_forecast.csv / .json / latest_forecast_metadata.json")
# reload checks (QC 22-23)
pd.read_csv(LIVE_DIR / "latest_block_forecast.csv")
json.loads((LIVE_DIR / "latest_block_forecast.json").read_text())
print("CSV + JSON reload: OK")


saved: latest_block_forecast.csv / .json / latest_forecast_metadata.json
CSV + JSON reload: OK


In [20]:
# Cell 21 — Human-readable forecast summary (6 blocks)
show = out[["block_name", "forecast_7d_total_rainfall_mm", "rainfall_category",
            "prob_low", "prob_normal", "prob_high", "max_daily_rainfall_mm",
            "max_daily_rainfall_day", "wet_days_in_next_7d", "dry_days_in_next_7d"]].copy()
print(f"7-day outlook issued {ISSUE_DATE.date()} (valid {TARGET_DATES[0].date()}..{TARGET_DATES[-1].date()}):")
print(show.to_string(index=False))


7-day outlook issued 2026-09-09 (valid 2026-09-10..2026-09-16):
block_name  forecast_7d_total_rainfall_mm rainfall_category  prob_low  prob_normal  prob_high  max_daily_rainfall_mm  max_daily_rainfall_day  wet_days_in_next_7d  dry_days_in_next_7d
     Dhuri                          22.50            NORMAL    0.2008       0.5280     0.2712                   7.57                       6                    4                    3
     Lehra                          22.81            NORMAL    0.1984       0.5253     0.2763                   9.29                       7                    3                    4
Malerkotla                          17.82            NORMAL    0.2534       0.5372     0.2094                   5.82                       6                    4                    3
    Moonak                          17.81            NORMAL    0.2534       0.5379     0.2087                   7.48                       7                    3                    4
   Sangrur           

In [21]:
# Cell 22 — Final readiness gate (prints READY FOR NOTEBOOK 6 only if all QC pass)
csv_p, json_p, meta_p = (LIVE_DIR / "latest_block_forecast.csv",
                         LIVE_DIR / "latest_block_forecast.json",
                         LIVE_DIR / "latest_forecast_metadata.json")
files_ok = all(p.exists() for p in [csv_p, json_p, meta_p])
extra = []
extra.append(("19. Output CSV exists", csv_p.exists()))
extra.append(("20. Output JSON exists", json_p.exists()))
extra.append(("21. Output metadata exists", meta_p.exists()))
try:
    rj = json.loads(json_p.read_text()); rc = pd.read_csv(csv_p)
    extra.append(("22. JSON reloads", isinstance(rj.get("blocks"), list) and len(rj["blocks"]) == 6))
    extra.append(("23. CSV reloads", len(rc) == 6))
except Exception as e:
    extra += [("22. JSON reloads", False), ("23. CSV reloads", False)]
    print(f"reload error: {e}")
extra.append(("24. row count exactly 6", len(out) == 6 and out["issue_date"].nunique() == 1))
for name, ok in extra:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
ALL_OK = QC_OK and files_ok and all(ok for _, ok in extra)
print()
if ALL_OK:
    print("=" * 60)
    print("NOTEBOOK 5 COMPLETE")
    print("=" * 60)
    print()
    print("Live inference: PASS")
    print("GEFS lead mapping: PASS")
    print("Six-block validation: PASS")
    print("Forecast generation: PASS")
    print("Probability generation: PASS")
    print("Threshold consistency: PASS")
    print("Leakage protection: PASS")
    print("Output CSV: PASS")
    print("Output JSON: PASS")
    print("Dashboard schema: PASS")
    print("Operational quality control: PASS")
    print()
    print(f"Forecast issue date: {ISSUE_DATE.date()}")
    print("Forecast horizon: 7 days")
    print("Blocks: 6")
    print("Selected method: Raw GEFS")
    print()
    print("READY FOR NOTEBOOK 6")
    print()
    print("=" * 60)
else:
    print("=" * 60)
    print("NOTEBOOK 5 NOT READY")
    print("=" * 60)
    for name, ok in qc:
        if not ok:
            print(f"  FAILED: {name}")
    if not files_ok:
        print("  FAILED: output files missing")
    raise SystemExit("Notebook 5 NOT READY — fix failed checks")


  [PASS] 19. Output CSV exists
  [PASS] 20. Output JSON exists
  [PASS] 21. Output metadata exists
  [PASS] 22. JSON reloads
  [PASS] 23. CSV reloads
  [PASS] 24. row count exactly 6

NOTEBOOK 5 COMPLETE

Live inference: PASS
GEFS lead mapping: PASS
Six-block validation: PASS
Forecast generation: PASS
Probability generation: PASS
Threshold consistency: PASS
Leakage protection: PASS
Output CSV: PASS
Output JSON: PASS
Dashboard schema: PASS
Operational quality control: PASS

Forecast issue date: 2026-09-09
Forecast horizon: 7 days
Blocks: 6
Selected method: Raw GEFS

READY FOR NOTEBOOK 6

